# ntv_double_selec

> 对应代码：`EconometricsCode/code_in_notes/Chap.10/ntv_double_selec.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.10`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.10")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


我们加载同一份 NTV 面板数据并声明面板结构，为使用滞后算子 L. 做准备。

In [2]:
%%stata
clear
set more off
use ../datasets/NTV_Aggregate_Data_reshaped.dta
tsset tik_id year


. clear

. set more off

. use ../datasets/NTV_Aggregate_Data_reshaped.dta

. tsset tik_id year

Panel variable: tik_id (strongly balanced)
 Time variable: year, 1995 to 2003, but with gaps
         Delta: 1 unit

. 


为了以更灵活的方式控制社会经济状况，我们把滞后对数人口与滞后对数工资各取 1 至 5 次多项式；循环一次生成 10 个高阶项，避免逐个手写。

In [3]:
%%stata
// 社会经济变量以及人口、工资的多项式
forvalues i=1/5{
    gen lag_log_pop_`i'=L.logpop^`i'
    gen lag_log_wage_`i'=L.log_wage^`i'
}


. // 社会经济变量以及人口、工资的多项式
. forvalues i=1/5{
  2.     gen lag_log_pop_`i'=L.logpop^`i'
  3.     gen lag_log_wage_`i'=L.log_wage^`i'
  4. }
(7,629 missing values generated)
(7,676 missing values generated)
(7,629 missing values generated)
(7,676 missing values generated)
(7,629 missing values generated)
(7,676 missing values generated)
(7,629 missing values generated)
(7,676 missing values generated)
(7,629 missing values generated)
(7,676 missing values generated)

. 


再补上护士数与人均医生数的一期滞后，连同两组多项式一起收入宏 E_controls，构成一个高维的候选控制变量集合。

In [4]:
%%stata
gen lag_nurses=L.nurses
gen lag_doctors_pc=L.doctors_pc
local E_controls "lag_log_pop_* lag_log_wage_* lag_nurses lag_doctors_pc"


. gen lag_nurses=L.nurses
(9,744 missing values generated)

. gen lag_doctors_pc=L.doctors_pc
(9,744 missing values generated)

. local E_controls "lag_log_pop_* lag_log_wage_* lag_nurses lag_doctors_pc"

. 


作为基准，我们先用 reghdfe 在 1999 年截面上做全控制回归：地区固定效应用 a(region) 吸收，标准误按地区聚类。这里没有做任何变量选择，所有多项式项一律进入回归。

In [5]:
%%stata
reghdfe Votes_SPS_ Watch_probit `E_controls' if year==1999, a(region) cluster(region)

(dropped 4 singleton observations)
(MWFE estimator converged in 1 iterations)

HDFE Linear regression                            Number of obs   =      2,050
Absorbing 1 HDFE group                            F(  13,     78) =      68.47
Statistics robust to heteroskedasticity           Prob > F        =     0.0000
                                                  R-squared       =     0.7704
                                                  Adj R-squared   =     0.7598
                                                  Within R-sq.    =     0.4355
Number of clusters (region)  =         79         Root MSE        =     2.0400

                                (Std. err. adjusted for 79 clusters in region)
------------------------------------------------------------------------------
             |               Robust
  Votes_SPS_ | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
Watch_probit | 

接着用 dsregress 做双重选择（double selection）Lasso：分别对结果方程与处理方程做 Lasso 选择，把两次被选中的控制变量取并集后再做 OLS，从而在遗漏重要控制的风险下仍对处理效应做有效推断；selec(cv) 表示用交叉验证选取惩罚参数，controls((i.region) ...) 把地区虚拟变量与多项式一起纳入被筛选的控制集。比较两个结果可以看出，变量选择与高维灵活控制会明显改变对 NTV 报道效应的估计与显著性。

In [6]:
%%stata
dsregress Votes_SPS_ Watch_probit if year==1999, controls((i.region) `E_controls') cluster(region) selec(cv)


Estimating lasso for Votes_SPS_ using cv
Estimating lasso for Watch_probit using cv

Double-selection linear model         Number of obs               =      2,054
                                      Number of controls          =         98
                                      Number of selected controls =         92
                                      Wald chi2(1)                =      31.79
                                      Prob > chi2                 =     0.0000

                                (Std. err. adjusted for 83 clusters in region)
------------------------------------------------------------------------------
             |               Robust
  Votes_SPS_ | Coefficient  std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
Watch_probit |   4.188653   .7429032     5.64   0.000     2.732589    5.644716
------------------------------------------------------------------------------
Note: Ch